In [1]:
import hoda
import tensorly as tl
random_state=42

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

fmin=1
fmax = 24
sfreq = 128

paradigm = P300(resample=sfreq, fmin=fmin, fmax=fmax)
dataset = BNCI2014008()
epochs, labels, meta = paradigm.get_data(
    dataset=dataset, 
    subjects=[3],
    return_epochs=True
)

<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(
BNCI2014008 has been renamed to BNCI2014_008. BNCI2014008 will be removed in version 1.1.
The dataset class name 'BNCI2014008' must be an abb

To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/base.py:354: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  X = mne.concatenate_epochs(X)


In [3]:
from hoda.hoda import BTTDA
from sklearn.pipeline import Pipeline
from hoda.classification import Vectorize, SelectF
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

hoda_params = dict(
    max_iter=128,
    tol=1e-12,
    init ='random',
    shrinkage='lw',
    toeplitz=None,
    obj='rt',
    solver='lanczos',
    taper=False,
    extra_train_info=False,
    verbose=False, 
    random_state=random_state,
    ortho=False
)

max_blocks = 8
pipeline = Pipeline([
    ('bttda', BTTDA(
        ranks=[None]*max_blocks,
        hoda_params=dict(
            **hoda_params,
            delta=0.1,
        ),
        extra_train_info=False,
        verbose=False
    )),
    ('vec', Vectorize()),
    ('zscore', StandardScaler()),
    ('select', SelectF(alpha=1)),
    ('clf', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])

In [4]:
from sklearn.model_selection import cross_validate, StratifiedKFold

X = tl.tensor(epochs.get_data())
y = labels
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

result = cross_validate(
    pipeline, X,y=y,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_estimator=True,
    return_indices=True
)

/tmp/ipykernel_8381/2799490432.py:3: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X = tl.tensor(epochs.get_data())
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


KeyboardInterrupt: 

In [ ]:
import math
from sklearn.metrics import roc_auc_score
import pandas as pd

block_results = []
for fold in range(cv.n_splits):
    estimator = result['estimator'][fold]
    train_idc = result['indices']['train'][fold]
    test_idc = result['indices']['test'][fold]
    Xt = estimator[:2].transform(X)
    for n_blocks in range(1, max_blocks+1):
        block_ranks = estimator[0].ranks_[:n_blocks]        
        n_features = sum([math.prod(ml_rank) for ml_rank in block_ranks])
        pipeline[2:].fit(Xt[train_idc,:n_features],y[train_idc])
        roc_auc = roc_auc_score(y[test_idc],pipeline[2:].decision_function(Xt[test_idc,:n_features]),)
        block_results.append(dict(fold=fold, n_blocks=n_blocks,roc_auc=roc_auc, n_features=n_features))
block_results = pd.DataFrame(block_results)

In [ ]:
from hoda.hoda import HODA
from sklearn.model_selection import GridSearchCV


hoda =Pipeline([
            ('hoda', HODA(**hoda_params)),
            ('vec', Vectorize()),
            ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')),
        ])
hoda_gs = GridSearchCV(
    hoda,
    param_grid=dict(hoda__rank=[1,2,4,8]),
    cv=cv,
    n_jobs=-1,
    refit=False
)
hoda_gs.fit(X,y)
hoda_gs.best_params_


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('default')
sns.lineplot(data=block_results, x='n_blocks', y='roc_auc')
#plt.axhline(hoda_gs.best_score_, color='red')

In [ ]:

max_features = X[0].size
for i in range(math.ceil(math.sqrt(max_features/max_blocks))):
    n = max_blocks*(i**2)
    plt.plot([0, max_blocks], [0, n], color='lightgray')
plt.axhline(max_features, color='red')
plt.axhline(min(X[0].shape)**2, color='green')

sns.lineplot(data=block_results, x='n_blocks', y='n_features')
